Today's topics:
* lists
* tuples
* dictionaries

# One sample

A thermoelectric material turns a temperature difference into voltage. How good it is at
that is one dimensionless number, the figure of merit zT, and bigger is better.

Tetrahedrites are copper antimony sulphosalts. They are cheap and earth-abundant next to
the tellurides they compete with, so people have been testing them for a decade to see
how high a zT they can reach.

Here is one sample out of that decade of work.

In [1]:
print('Heo 2014, composition Cu11 Mn, Mn-doped, zT 1.13 at 302 C')

Heo 2014, composition Cu11 Mn, Mn-doped, zT 1.13 at 302 C


A study name, a composition, a dopant family, a number, and a temperature. Two of those
are text and three are numbers.

*What happens if we put the whole thing in an array?*

In [2]:
import numpy as np

sample = np.array(['Heo 2014', 'Cu11 Mn', 'Mn', 1.13, 302])
print(sample)
print(sample.dtype)

['Heo 2014' 'Cu11 Mn' 'Mn' '1.13' '302']
<U32


Everything became a string. The `dtype` starts with `<U`, NumPy's label for text.

There was no error and no warning. `1.13` is now the characters `1`, `.`, `1`, `3`, and
we can't do arithmetic with it.

In [3]:
print(sample[3] * 2)

1.131.13


Not 2.26. Multiplying a string by 2 repeats it, so the zT of a sample came back as
`1.131.13` with no complaint from anyone.

Arithmetic that has no string meaning does fail:

In [4]:
# EXPECTED-ERROR: this cell fails on purpose -- see the surrounding text
sample[3] - sample[4]

TypeError: unsupported operand type(s) for -: 'numpy.str_' and 'numpy.str_'

> Read a traceback from the bottom. The error type is the diagnosis and the last line is
> where it happened. Both are in there before anyone has to be asked.

An array holds one dtype for everything in it. That's what makes its arithmetic fast,
and it's why every array we've built so far has been all numbers.

A measurement isn't all numbers. Today we cover the containers that hold the rest: lists,
tuples and dictionaries.

In [5]:
#@title Load the tetrahedrite thermoelectric database (click ▶ to run, data loading, not a learning objective) { display-mode: "form" }
import os

import pandas as pd

_file = 'thermoelectrics.csv'
_github = f'https://raw.githubusercontent.com/wfreinhart/matse219/main/datasets/{_file}'
_local = next(
    (p for p in (f'datasets/{_file}', f'../datasets/{_file}', f'../../datasets/{_file}',
                 f'../../../datasets/{_file}')
     if os.path.exists(p)),
    None,
)

try:
    _table = pd.read_csv(_local if _local else _github)
except Exception as e:
    raise RuntimeError(
        f"Could not load '{_file}'. If you are in Colab, check your internet "
        f"connection and that the file exists at {_github}"
    ) from e

_rows = _table.dropna(subset=['zT_max'])
_by_study = _rows.groupby('study_id')['zT_max']

studies = [str(name) for name in _by_study.max().index]
best_zt = [round(float(v), 2) for v in _by_study.max()]
n_samples = [int(v) for v in _by_study.size()]

zt_kosaka = [round(float(v), 2) for v in _rows[_rows['study_id'] == 'Kosaka 2017']['zT_max']]
zt_harish = [round(float(v), 2) for v in _rows[_rows['study_id'] == 'Harish 2016']['zT_max']]

print(f'Loaded {len(_rows)} tetrahedrite samples from {len(studies)} studies')
print(f'  studies:   {len(studies)} names, e.g. {studies[0]!r}')
print(f'  best_zt:   {len(best_zt)} values, highest {max(best_zt)}')
print(f'  n_samples: {len(n_samples)} counts, from {min(n_samples)} to {max(n_samples)}')

Loaded 278 tetrahedrite samples from 63 studies
  studies:   63 names, e.g. 'Ahn 2021'
  best_zt:   63 values, highest 1.13
  n_samples: 63 counts, from 1 to 11


# Lists

The loader handed back two **lists**: the name of every study, and the highest zT that
study reported.

In [6]:
print(type(studies))
print(studies[0])
print(best_zt[0])

<class 'list'>
Ahn 2021
0.66


Square brackets, no `np.`. We've been writing lists since L02, inside every
`np.array([...])`; we just never named them.

`studies` holds text and `best_zt` holds floats, and neither was converted. The two are
parallel in the sense L03 used: position 0 is the same study in both.

## Indexing, slicing, length

Everything we learned on arrays works here:

In [7]:
print(studies[0])
print(studies[-1])
print(len(studies))

Ahn 2021
Zhu 2022
63


In [8]:
print(studies[:3])

['Ahn 2021', 'Balaz_2020', 'Barbier 2015']


63 studies, alphabetical, and a slice takes a contiguous run of them.

## Looking something up by name

We rarely know a study's position. We know its name.

In [9]:
i = studies.index('Heo 2014')
print(i)
print(best_zt[i])

13
1.13


`.index()` searches the list by value and returns the position, which we then use in the
parallel list. That's 1.13, the highest zT in this whole database, from the sample we
opened with.

> This is why the two lists must stay in the same order. Sorting one and not the other
> would attach every zT to the wrong study, and nothing would raise an error.

## Lists can change

An array can't change its length. A list can.

In [10]:
studies.append('Reinhart 2026')
best_zt.append(0.42)
print(len(studies), len(best_zt))

64 64


`.append()` adds one entry at the end, and here it has to be done to both lists to keep
them aligned.

We can also replace what sits at a position:

In [11]:
best_zt[-1] = 0.55
print(best_zt[-1])

0.55


### [Check your understanding]

1. Print the number of studies, then the last three study names using a slice.
2. Use `.index()` to find `'Lu 2015'`, and print its best zT from `best_zt`.
3. `.append()` one made-up study name to `studies` and one number to `best_zt`, then
   print the length of each.
4. Print the first entry of `n_samples`, which counts how many samples that study
   reported.

*What would go wrong if you appended to `studies` and forgot `best_zt`?*

## Ragged rows

Studies don't report the same number of samples. Kosaka 2017 reported eleven; Harish
2016 reported one.

In [12]:
print(len(zt_kosaka), len(zt_harish))
print(zt_kosaka)
print(zt_harish)

11 1
[0.44, 0.66, 0.67, 0.63, 0.59, 0.59, 0.65, 0.65, 0.65, 0.57, 0.57]
[0.04]


Two lists of different lengths. Stacking them into one array is the natural next thought,
and it fails:

In [13]:
# EXPECTED-ERROR: this cell fails on purpose -- see the surrounding text
np.array([zt_kosaka, zt_harish])

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (2,) + inhomogeneous part.

`ValueError`, and the message says *inhomogeneous shape*. An array has to be a rectangle:
every row the same length.

Real studies aren't rectangular. Earlier the array refused mixed types; here it refuses
ragged lengths, and a list of lists is what holds them.

# Arithmetic on a list

The zT values in Kosaka's study are ordinary numbers. *Can we scale them all at once, the
way we would with an array?*

In [14]:
# EXPECTED-ERROR: this cell fails on purpose -- see the surrounding text
zt_kosaka * 1.1

TypeError: can't multiply sequence by non-int of type 'float'

`TypeError`. A list is a container, not an array, and Python won't guess that we meant
each entry.

Multiplying by a whole number doesn't raise an error, which is worse:

In [15]:
print(len(zt_kosaka * 2))

22


Twenty-two entries: the same eleven numbers twice, end to end. On a list, `*` repeats the
sequence.

The fix is the conversion we've been writing since L02:

In [16]:
zt_array = np.array(zt_kosaka)
print(np.max(zt_array))
print(np.median(zt_array))

0.67
0.63


Once it's an array, every statistic from L05 and L06 is available again. Lists hold
things, including things arrays refuse; arrays do arithmetic.

## Tuples

One more container, which we mostly receive rather than write.

In [17]:
grid = np.array([[0.44, 0.66, 0.67], [0.63, 0.59, 0.59]])
shape = grid.shape
print(shape)
print(type(shape))
print(shape[0])

(2, 3)
<class 'tuple'>
2


`.shape` has been handing us a tuple since L04. It indexes like a list and it is
**immutable**: once built, entries can't be replaced.

In [18]:
# EXPECTED-ERROR: this cell fails on purpose -- see the surrounding text
shape[0] = 5

TypeError: 'tuple' object does not support item assignment

`TypeError`. That is the point of a tuple: an array's shape shouldn't change because
someone assigned to it.

# Dictionaries

The parallel lists work, and they have one weakness. Here is position 13 in each:

In [19]:
print(studies[13], best_zt[13], n_samples[13])

Heo 2014 1.13 9


Three lists, one position, and the meaning of each number lives only in the variable
name. Add a fourth measurement and there is a fourth list to keep aligned.

A **dictionary** stores each value under a name we choose.

In [20]:
study = {'name': 'Heo 2014', 'dopant': 'Mn', 'best_zt': 1.13, 'n_samples': 9}
print(study)

{'name': 'Heo 2014', 'dopant': 'Mn', 'best_zt': 1.13, 'n_samples': 9}


Curly brackets, and `name: value` pairs separated by commas. The names are called
**keys**, and here they're strings. Each key appears once: assigning to a key that's
already there replaces its value rather than adding a second copy.

`len()` counts pairs:

In [21]:
print(len(study))

4


We read a value by naming its key:

In [22]:
print(study['best_zt'])
print(study['dopant'])

1.13
Mn


We didn't count positions, and there's no doubt about which number we asked for. One
record now travels as one object, with the study that produced it attached.

## Adding to a dictionary

Assigning to a key that doesn't exist yet creates it:

In [23]:
study['temperature_C'] = 302
study['composition'] = 'Cu11 Mn'
print(study)

{'name': 'Heo 2014', 'dopant': 'Mn', 'best_zt': 1.13, 'n_samples': 9, 'temperature_C': 302, 'composition': 'Cu11 Mn'}


The record now holds two strings, two floats and an integer, each under its own name.
This is what the array could not do.

## When the key is not there

In [24]:
# EXPECTED-ERROR: this cell fails on purpose -- see the surrounding text
print(study['seebeck'])

KeyError: 'seebeck'

`KeyError`, and the message is the key we asked for. Nearly always a typo, or a field
that was never added.

A list refuses a position that doesn't exist. A dictionary creates the key instead, which
is why a misspelled key is silent when we write and loud when we read.

`.keys()` reports what a record actually holds. It prints as `dict_keys([...])` rather
than a plain list because it's a live view: add a key and the same view shows it.

In [25]:
print(study.keys())

dict_keys(['name', 'dopant', 'best_zt', 'n_samples', 'temperature_C', 'composition'])


## Computing from a record

The values are ordinary numbers, so ordinary arithmetic works:

In [26]:
zt = study['best_zt']
print(f"{study['name']} reached zT = {zt:.2f} at {study['temperature_C']} C")

Heo 2014 reached zT = 1.13 at 302 C


### [Check your understanding]

Lu 2015 reported 4 samples and a best zT of 1.03.

1. Build a dictionary called `lu` with keys `'name'`, `'best_zt'` and `'n_samples'`.
2. Print its best zT by naming the key.
3. Add a `'dopant'` key holding any string, then print `lu.keys()`.
4. Print how many pairs `lu` holds.

*`study['seebeck']` failed above. What would `lu['Best_zt']` do, and why?*

# Many records

One dictionary is one study. A list of dictionaries is a small table.

In [27]:
top = [
    {'name': 'Heo 2014', 'n_samples': 9, 'best_zt': 1.13},
    {'name': 'Yan  2018', 'n_samples': 1, 'best_zt': 1.10},
    {'name': 'Lu 2015', 'n_samples': 4, 'best_zt': 1.03},
    {'name': 'Yan 2018', 'n_samples': 6, 'best_zt': 1.00},
]
print(len(top))

4


Two steps to reach a value: position in the list, then key in the dictionary.

In [28]:
print(top[0]['name'], top[0]['best_zt'])
print(top[-1]['name'], top[-1]['best_zt'])

Heo 2014 1.13
Yan 2018 1.0


Four studies within 0.13 of each other, and the top one rests on nine samples while the
second rests on one.

That is the argument for carrying records rather than bare numbers. A zT of 1.10 on its
own is a number someone can quote. With its record attached it is one sample from one
study, and a reader can go and find it.

### [Check your understanding]

1. Print the `n_samples` of the second entry in `top`.
2. Build a record for Lu 2013, which reported 4 samples and a best zT of 1.00, and
   `.append()` it to `top`.
3. Print `len(top)` and the name in the last record.
4. Add a `'source'` key to your new record holding the string `'tetrahedrite database'`.

*Two of these studies are called Yan 2018. What does that tell you about using a name as
an identifier?*

Writing one line per study is fine for four of them and hopeless for 63. Next lecture we
read a whole table in one call, and the names stay attached to the columns.

In [29]:
#@title Optional tool glimpse: pandas turns a list of records into a table (click ▶ to run) { display-mode: "form" }
import pandas as pd

pd.DataFrame(top)

,name,n_samples,best_zt
0,Heo 2014,9,1.13
1,Yan 2018,1,1.10
2,Lu 2015,4,1.03
3,Yan 2018,6,1.00


# Summary

* A **list** holds things in order, of any type, and it can grow or be edited.
* A **tuple** is an ordered sequence whose entries can't be replaced once it's built,
  which is why `.shape` is one.
* A **dictionary** holds values under names, so nothing depends on counting positions.
* A numeric **array** holds one dtype in a rectangle, and supports elementwise
  arithmetic in exchange.

Parallel lists and a dictionary hold the same information. The dictionary keeps the names
attached to it, which is what lets a measurement travel without losing what it was.

## Further reading

* VanderPlas, *A Whirlwind Tour of Python*, "Built-In Data Structures"
* The Python tutorial, [Data Structures](https://docs.python.org/3/tutorial/datastructures.html)
* The data: [Tetrahedrite Thermoelectrics Database](https://doi.org/10.5281/zenodo.17692053), CC BY 4.0